In [23]:
import json
import cv2
import random
from pathlib import Path
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

%matplotlib inline


## 0.1 Paths

`pvsg.json` (3.88MB) can be downloaded directly from
https://huggingface.co/datasets/Jingkang/PVSG/resolve/main/pvsg.json --
no need to clone the whole repo or download the 10.8GB of masks/videos just
for annotation-level exploration.

Mirroring OpenPVSG's own expected structure (so any of their scripts still
work unmodified if you use them later), assumed layout:

```
master-degree-project/
└── data/PVSG_Dataset/
    ├── pvsg.json
    └── data/
        ├── vidor/{frames, masks, videos}
        ├── epic_kitchen/{frames, masks, videos}
        └── ego4d/{frames, masks, videos}
```

Adjust `PROJECT_ROOT` below if this notebook doesn't sit at
`src/notebooks/`, same as the ASPIRe notebook.


In [20]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]

VIDOR_ROOT = PROJECT_ROOT / "data" / "ViDOR"
VIDEO_TRAIN_DATA = VIDOR_ROOT / "train" / "video"
TRAIN_DATA_JSON = VIDOR_ROOT / "train_files.json"
TRAIN_DATA_ANNOTATIONS = VIDOR_ROOT / "training_annotation"  # frames/masks/videos per source live under here

print("PROJECT_ROOT   :", PROJECT_ROOT, "-> exists:", PROJECT_ROOT.exists())
print("VIDOR_ROOT      :", VIDOR_ROOT, "-> exists:", VIDOR_ROOT.exists())
print("VIDEO_TRAIN_DATA      :", VIDEO_TRAIN_DATA, "-> exists:", VIDEO_TRAIN_DATA.exists())
print("TRAIN_DATA :", TRAIN_DATA_JSON, "-> exists:", TRAIN_DATA_JSON.exists())
print("TRAIN_DATA_ANNOTATIONS :", TRAIN_DATA_ANNOTATIONS, "-> exists:", TRAIN_DATA_ANNOTATIONS.exists())

# assert PVSG_JSON.exists(), (
#     "pvsg.json not found -- download it from "
#     "https://huggingface.co/datasets/Jingkang/PVSG/resolve/main/pvsg.json "
#     "and place it at the PVSG_JSON path above before continuing."
# )


PROJECT_ROOT   : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project -> exists: True
VIDOR_ROOT      : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\ViDOR -> exists: True
VIDEO_TRAIN_DATA      : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\ViDOR\train\video -> exists: True
TRAIN_DATA : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\ViDOR\train_files.json -> exists: True
TRAIN_DATA_ANNOTATIONS : C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\data\ViDOR\training_annotation -> exists: True


In [61]:
OUTPUT_PATH = PROJECT_ROOT / "outputs" / "notebooks" / "20260725_ViDOR_dataset_exploration"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

output_video = OUTPUT_PATH / "2401075277_visualization.mp4"

In [57]:
path_to_video_annotation = TRAIN_DATA_ANNOTATIONS / "0027" / "2530259622.json"
video_path = VIDEO_TRAIN_DATA / "0027" / "2530259622.mp4"

with open(path_to_video_annotation, "r") as f:
    annotation = json.load(f)

print(f"Video ID     : {annotation['video_id']}")
print(f"Frame count  : {annotation['frame_count']}")
print(f"FPS          : {annotation['fps']}")
print(f"Resolution   : {annotation['width']} x {annotation['height']}")
print()

# tid -> category
tid_to_category = {
    obj["tid"]: obj["category"]
    for obj in annotation["subject/objects"]
}

trajectories = annotation["trajectories"]
relations = annotation["relation_instances"]

print("Objects in video:")
for tid, category in tid_to_category.items():
    print(f"  tid={tid}: {category}")




Video ID     : 2530259622
Frame count  : 477
FPS          : 29.97002997002997
Resolution   : 640 x 480

Objects in video:
  tid=0: child
  tid=1: child
  tid=2: child
  tid=3: car
  tid=4: car
  tid=5: car
  tid=6: car
  tid=7: bicycle
  tid=8: car
  tid=9: bicycle
  tid=10: bicycle


# Object + Relations Tracking

In [62]:
# ==============================================================================
# Video
# ==============================================================================

cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError("Cannot open video.")

frame_idx = 0

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# MP4 codec
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(output_video),
    fourcc,
    fps,
    (width, height),
)

# ==============================================================================
# Main loop
# ==============================================================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_annotations = trajectories[frame_idx]

    # -------------------------------------------------------------------------
    # Find active relations for this frame
    # -------------------------------------------------------------------------

    active_relations = []

    for rel in relations:

        if rel["begin_fid"] <= frame_idx < rel["end_fid"]:
            active_relations.append(rel)

    # -------------------------------------------------------------------------
    # Group relations by SUBJECT
    #
    # Example:
    #
    # person (0)
    #     next_to, behind -> chair (1)
    #     towards -> toy (2)
    #
    # -------------------------------------------------------------------------

    relations_by_subject = defaultdict(lambda: defaultdict(list))

    for rel in active_relations:

        s_tid = rel["subject_tid"]
        o_tid = rel["object_tid"]

        predicate = rel["predicate"]
        object_name = tid_to_category[o_tid]

        relations_by_subject[s_tid][f"{object_name} ({o_tid})"].append(predicate)

    # -------------------------------------------------------------------------
    # Draw objects
    # -------------------------------------------------------------------------

    for obj in frame_annotations:

        tid = obj["tid"]

        category = tid_to_category[tid]

        bbox = obj["bbox"]

        xmin = bbox["xmin"]
        ymin = bbox["ymin"]
        xmax = bbox["xmax"]
        ymax = bbox["ymax"]

        generated = obj["generated"]

        color = (0, 255, 0) if generated == 0 else (0, 255, 255)

        # -------------------------------------------------------------
        # Bounding box
        # -------------------------------------------------------------

        cv2.rectangle(
            frame,
            (xmin, ymin),
            (xmax, ymax),
            color,
            2,
        )

        # -------------------------------------------------------------
        # Object name
        # -------------------------------------------------------------

        cv2.putText(
            frame,
            f"{category} ({tid})",
            (xmin, ymin - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.60,
            color,
            2,
            cv2.LINE_AA,
        )

        # -------------------------------------------------------------
        # Relations
        # -------------------------------------------------------------

        if tid in relations_by_subject:

            relation_lines = []

            for target, predicates in relations_by_subject[tid].items():

                text = f"{', '.join(predicates)} -> {target}"

                relation_lines.append(text)

            # Start high enough so all relations fit above the bbox
            y = ymin - 25 - 18 * len(relation_lines)

            for line in relation_lines:

                # white background rectangle

                (w, h), baseline = cv2.getTextSize(
                    line,
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    1,
                )

                cv2.rectangle(
                    frame,
                    (xmin - 2, y - h - 2),
                    (xmin + w + 2, y + baseline),
                    (255, 255, 255),
                    -1,
                )

                cv2.putText(
                    frame,
                    line,
                    (xmin, y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    (0, 0, 0),
                    1,
                    cv2.LINE_AA,
                )

                y += 18

    # -------------------------------------------------------------------------
    # Frame number
    # -------------------------------------------------------------------------

    cv2.putText(
        frame,
        f"Frame: {frame_idx}",
        (10, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 0, 255),
        2,
    )

    # -------------------------------------------------------------------------
    # Show
    # -------------------------------------------------------------------------
    writer.write(frame)
    
    cv2.imshow("VIDOR Scene Graph", frame)

    key = cv2.waitKey(30) & 0xFF

    if key == 27:      # ESC
        break

    elif key == ord(" "):  # Pause
        cv2.waitKey(0)

    frame_idx += 1

cap.release()
writer.release()
cv2.destroyAllWindows()

print(f"Saved visualization to:\n{output_video}")

Saved visualization to:
C:\Users\Samuel Oliveira\Desktop\CS\master-degree-project\outputs\notebooks\20260725_ViDOR_dataset_exploration\2401075277_visualization.mp4
